In [38]:
from collections import defaultdict, Counter

In [39]:

class SemanticRouter:

    def __init__(self, threshold=0.8):
        self.threshold = threshold
        # { route_name: [utterance, ...] }
        self.routes: dict[str, list[str]] = {}


    def _cosine_similarity(self, vec1: list[float], vec2: list[float]) -> float:
        """Cosine similarity between two equal-length vectors."""
        dot_product = sum(a * b for a, b in zip(vec1, vec2))
        mag1 = sum(a ** 2 for a in vec1) ** 0.5
        mag2 = sum(b ** 2 for b in vec2) ** 0.5
        if mag1 == 0 or mag2 == 0:
            return 0.0
        return dot_product / (mag1 * mag2)

    def _build_vocab(self, *texts: str) -> list[str]:
        """Build a sorted vocabulary from all provided texts."""
        words = set()
        for text in texts:
            words.update(text.lower().split())
        return sorted(words)

    def _vectorize(self, text: str, vocab: list[str]) -> list[int]:
        """Encode *text* as a bag-of-words vector over *vocab*."""
        counts = Counter(text.lower().split())
        return [counts.get(word, 0) for word in vocab]

    def _all_utterances(self) -> list[tuple[str, str]]:
        """Return a flat list of (route_name, utterance) pairs."""
        return [
            (route, utt)
            for route, utterances in self.routes.items()
            for utt in utterances
        ]

    def add_route(self, route_name: str, utterances: list[str]) -> None:
        if route_name in self.routes:
            self.routes[route_name].extend(utterances)
        else:
            self.routes[route_name] = list(utterances)

    def remove_route(self, route_name: str) -> None:
        """Delete a route entirely."""
        self.routes.pop(route_name, None)

    def list_routes(self) -> dict[str, list[str]]:
        """Return all registered routes and their utterances."""
        return dict(self.routes)

    def route(self, query: str) -> str | None:
        pairs = self._all_utterances()
        if not pairs:
            return None

        all_texts = [query] + [utt for _, utt in pairs]
        vocab = self._build_vocab(*all_texts)

        query_vec = self._vectorize(query, vocab)

        scores = [
            self._cosine_similarity(query_vec, self._vectorize(utt, vocab))
            for _, utt in pairs
        ]

        best_idx = scores.index(max(scores))
        if scores[best_idx] >= self.threshold:
            return pairs[best_idx]

        return None

In [40]:
router = SemanticRouter(threshold=0.3)

router.add_route("greeting",  ["good morning there", "good morning", "hei, hi how are you"])
router.add_route("billing",   ["invoice", "payment due", "how much do I owe"])
router.add_route("technical", ["bug report", "error in my code", "app is crashing"])


In [41]:

router.route("hi, good morning")


('greeting', 'good morning')

In [42]:
router._all_utterances()

[('greeting', 'good morning there'),
 ('greeting', 'good morning'),
 ('greeting', 'hei, hi how are you'),
 ('billing', 'invoice'),
 ('billing', 'payment due'),
 ('billing', 'how much do I owe'),
 ('technical', 'bug report'),
 ('technical', 'error in my code'),
 ('technical', 'app is crashing')]